# Kaggle 95-Cloud training orchestrator

Notebook nay chi cau hinh va goi pipeline chuan trong `src/`. Dataset, model, training loop, metric va ONNX export khong duoc dinh nghia lai tai day.

Luong chay: `tests -> preprocess_95cloud.py -> split_dataset.py -> CloudDataset validation -> train.py -> eval.py -> export_onnx.py`.

> Checkpoint duoc tao boi phien ban notebook cu la **legacy**. Notebook nay khong nap checkpoint legacy va khong dung no de export artifact production. Baseline hien tai van bi chan cho cross-satellite cho den khi radiometric contract cua 95-Cloud duoc xac minh.

## 1. Dependencies

Kaggle image can co san `torch` va `torchvision`. Cell nay chi cai cac package con thieu, bao gom `wandb` de theo doi training run.

In [ ]:
import importlib.util
import subprocess
import sys

required_packages = {
    "torch": "torch",
    "torchvision": "torchvision",
    "tifffile": "tifffile",
    "tqdm": "tqdm",
    "onnx": "onnx",
    "kagglehub": "kagglehub",
    "wandb": "wandb",
}
missing = [package for module, package in required_packages.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are available.")

## 2. Experiment configuration

`channels=4` la contract mac dinh cua pipeline 95-Cloud (RGB+NIR). Dat `channels=3` neu can checkpoint RGB rieng; khi do `channel_dropout_p` phai bang `0`. Mac dinh notebook clone nhanh `repo_url` tai `repo_branch` vao `/kaggle/working/repo_dir`; `project_root` chi dung de override khi debug.

In [ ]:
from pathlib import Path
import hashlib
import importlib.metadata
import json
import os
import platform
import shlex

import torch

CFG = {
    "repo_url": "https://github.com/hoxuanphu/cube_nano.git",
    "repo_branch": "main",
    "repo_dir": "cube_nano",
    "clone_repo": True,
    "project_root": None,
    "dataset_root": None,
    "kaggle_dataset_slug": "sorour/95cloud-cloud-segmentation-on-satellite-images",
    "download_dataset_if_missing": True,
    "work_dir": None,
    "seed": 42,
    "channels": 4,
    "source_patch_size": 384,
    "crop_size": 256,
    "cloud_ratio_threshold": 0.50,
    "probability_threshold": 0.50,
    "val_ratio": 0.15,
    "test_ratio": 0.15,
    "epochs": 12,
    "batch_size": 32,
    "learning_rate": 1e-3,
    "num_workers": 0,
    "channel_dropout_p": 0.3,
    "use_pos_weight": True,
    "pos_weight_samples_per_patch": 8,
    "amp": True,
    "use_wandb": True,
    "wandb_project": "cube-nano",
    "wandb_entity": None,
    "wandb_run_name": None,
    "wandb_group": "95-cloud",
    "wandb_tags": ["kaggle", "95-cloud", "mobilenetv3-small"],
    "wandb_mode": "online",
    "wandb_api_key_secret": "WANDB_API_KEY",
    "run_tests": True,
    "run_preprocess": True,
    "force_preprocess": False,
    "run_split": True,
    "force_split": True,
    "validate_dataset": True,
    "dataset_validation_batch_size": 64,
    "run_train": True,
    "run_eval": True,
    "run_export": True,
    "dynamic_batch_onnx": False,
}

if CFG["channels"] not in (3, 4):
    raise ValueError("channels must be 3 (RGB) or 4 (RGB+NIR).")
if CFG["channels"] == 3 and CFG["channel_dropout_p"] != 0:
    raise ValueError("channel_dropout_p must be 0 for a 3-channel RGB model.")
if CFG["dataset_validation_batch_size"] <= 0:
    raise ValueError("dataset_validation_batch_size must be greater than zero.")
if CFG["wandb_mode"] not in ("online", "offline", "disabled"):
    raise ValueError("wandb_mode must be online, offline or disabled.")
if CFG["use_wandb"] and not CFG["wandb_project"]:
    raise ValueError("wandb_project must not be empty when W&B is enabled.")

# The actual training seed is applied by src/train.py. Keep the notebook kernel
# deterministic as well so provenance never records benchmark mode as enabled.
os.environ["PYTHONHASHSEED"] = str(CFG["seed"])
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

IS_KAGGLE = Path("/kaggle").is_dir()
INPUT_ROOT = Path("/kaggle/input") if IS_KAGGLE else Path.cwd()
DEFAULT_WORK_ROOT = Path("/kaggle/working") if IS_KAGGLE else Path.cwd()

if CFG["use_wandb"] and CFG["wandb_mode"] == "online":
    if IS_KAGGLE and not os.environ.get("WANDB_API_KEY"):
        try:
            from kaggle_secrets import UserSecretsClient
            os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret(
                CFG["wandb_api_key_secret"]
            )
        except Exception as exc:
            raise RuntimeError(
                "W&B is enabled but the Kaggle Secret WANDB_API_KEY is unavailable. "
                "Add it under Add-ons > Secrets, or set wandb_mode='offline'."
            ) from exc
    print("W&B online logging enabled.")
print(json.dumps(CFG, indent=2))

## 3. Resolve project and command runner

Bat Internet cho Kaggle Notebook. Cell nay clone repo GitHub vao `/kaggle/working`, cap nhat bang fast-forward khi chay lai, sau do dung truc tiep cac entrypoint trong `src/`.

In [ ]:
REQUIRED_ENTRYPOINTS = (
    "src/data/preprocess_95cloud.py",
    "src/data/split_dataset.py",
    "src/data/cloud_dataset.py",
    "src/train.py",
    "src/models/mobilenetv3.py",
    "src/eval.py",
    "src/export_onnx.py",
)

def is_project_root(path):
    path = Path(path)
    return all((path / relative_path).is_file() for relative_path in REQUIRED_ENTRYPOINTS)

def clone_or_update_project():
    repo_url = CFG["repo_url"].strip()
    repo_branch = CFG["repo_branch"].strip()
    repo_dir = DEFAULT_WORK_ROOT / CFG["repo_dir"]
    if not repo_url or not repo_branch:
        raise ValueError("repo_url and repo_branch must not be empty.")

    if repo_dir.exists():
        if not (repo_dir / ".git").is_dir():
            raise FileExistsError(
                f"Clone target exists but is not a Git repository: {repo_dir}"
            )
        print(f"Updating repository: {repo_dir}")
        subprocess.run(
            ["git", "-C", str(repo_dir), "fetch", "--depth", "1", "origin", repo_branch],
            check=True,
        )
        subprocess.run(
            ["git", "-C", str(repo_dir), "checkout", repo_branch],
            check=True,
        )
        subprocess.run(
            ["git", "-C", str(repo_dir), "pull", "--ff-only", "origin", repo_branch],
            check=True,
        )
    else:
        print(f"Cloning {repo_url} ({repo_branch}) -> {repo_dir}")
        subprocess.run(
            [
                "git", "clone", "--depth", "1", "--branch", repo_branch,
                repo_url, str(repo_dir),
            ],
            check=True,
        )

    if not is_project_root(repo_dir):
        raise FileNotFoundError(f"Cloned repository is missing required src files: {repo_dir}")
    return repo_dir.resolve()

def find_project_root():
    if CFG["project_root"]:
        candidate = Path(CFG["project_root"]).expanduser().resolve()
        if not is_project_root(candidate):
            raise FileNotFoundError(f"Configured project_root is incomplete: {candidate}")
        return candidate

    if CFG["clone_repo"]:
        return clone_or_update_project()

    direct_candidates = [Path.cwd(), *Path.cwd().parents, DEFAULT_WORK_ROOT]
    for candidate in direct_candidates:
        if is_project_root(candidate):
            return candidate.resolve()

    search_roots = [root for root in (DEFAULT_WORK_ROOT, INPUT_ROOT) if root.is_dir()]
    for search_root in search_roots:
        for train_script in sorted(search_root.rglob("src/train.py")):
            candidate = train_script.parent.parent
            if is_project_root(candidate):
                return candidate.resolve()
    raise FileNotFoundError(
        "Could not locate the project. Attach the repository to Kaggle or set CFG['project_root']."
    )

PROJECT_ROOT = find_project_root()
RUN_ROOT = (
    Path(CFG["work_dir"]).expanduser().resolve()
    if CFG["work_dir"]
    else (DEFAULT_WORK_ROOT / "cloud_95cloud_run").resolve()
)
PROCESSED_ALL = RUN_ROOT / "data" / "processed" / "all"
PROCESSED_ROOT = RUN_ROOT / "data" / "processed"
CHECKPOINT_DIR = RUN_ROOT / "checkpoints"
RESULTS_DIR = RUN_ROOT / "results"
LOG_DIR = RUN_ROOT / "logs"
BEST_CHECKPOINT = CHECKPOINT_DIR / "best_model.pth"
ONNX_PATH = CHECKPOINT_DIR / "cloud_model.onnx"
for directory in (RUN_ROOT, CHECKPOINT_DIR, RESULTS_DIR, LOG_DIR):
    directory.mkdir(parents=True, exist_ok=True)

def print_log_tail(log_path, max_lines=40):
    lines = log_path.read_text(encoding="utf-8", errors="replace").splitlines()
    omitted = max(0, len(lines) - max_lines)
    if omitted:
        print(f"... {omitted:,} earlier log lines omitted from notebook output ...")
    if lines:
        print("\n".join(lines[-max_lines:]))

def run_python(label, script_or_module, *arguments, module=False, cwd=None):
    command = [sys.executable]
    if module:
        command.extend(["-m", str(script_or_module)])
    else:
        command.append(str(PROJECT_ROOT / script_or_module))
    command.extend(str(argument) for argument in arguments)
    environment = os.environ.copy()
    environment["PYTHONHASHSEED"] = str(CFG["seed"])
    environment["PYTHONUNBUFFERED"] = "1"
    log_slug = "".join(character.lower() if character.isalnum() else "_" for character in label).strip("_")
    log_path = LOG_DIR / f"{log_slug}.log"
    print(f"\n[{label}]")
    print("$", shlex.join(command))
    print("Full log:", log_path)
    with log_path.open("w", encoding="utf-8") as log_file:
        completed = subprocess.run(
            command,
            cwd=str(cwd or RUN_ROOT),
            env=environment,
            stdout=log_file,
            stderr=subprocess.STDOUT,
            check=False,
        )
    print_log_tail(log_path)
    if completed.returncode != 0:
        raise subprocess.CalledProcessError(completed.returncode, command)

print("Project root:", PROJECT_ROOT)
print("Run root:", RUN_ROOT)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 4. Resolve 95-Cloud data

Quet TIFF de quy va nhan dien band tu ten file hoac thu muc cha. Ho tro ca layout `train_*` va cac layout long thu muc cua Kaggle.

In [ ]:
src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
from data.preprocess_95cloud import discover_scene_files

def validate_95cloud_root(path):
    path = Path(path).expanduser().resolve()
    scenes = discover_scene_files(path, channels=CFG["channels"])
    print(f"Discovered {len(scenes)} complete 95-Cloud scenes under {path}")
    return path

def resolve_95cloud_root():
    if CFG["dataset_root"]:
        return validate_95cloud_root(CFG["dataset_root"])

    owner, dataset_name = CFG["kaggle_dataset_slug"].split("/", 1)
    attached_candidates = (
        INPUT_ROOT / dataset_name,
        INPUT_ROOT / "datasets" / owner / dataset_name,
    )
    for candidate in attached_candidates:
        if candidate.is_dir():
            return validate_95cloud_root(candidate)
    if not CFG["download_dataset_if_missing"]:
        raise FileNotFoundError(
            f"95-Cloud is not attached. Checked: {[str(path) for path in attached_candidates]}"
        )

    import kagglehub
    downloaded = Path(kagglehub.dataset_download(CFG["kaggle_dataset_slug"]))
    return validate_95cloud_root(downloaded)

DATA_ROOT = resolve_95cloud_root()
print("95-Cloud root:", DATA_ROOT)
print("Band order:", ["red", "green", "blue"] + (["nir"] if CFG["channels"] == 4 else []))
print("Radiometric contract: UNVERIFIED - baseline use only")

## 5. Run regression tests

Chay lai test noisy-label va scene split truoc khi tao artifact moi. Loi test se dung notebook ngay lap tuc.

In [ ]:
if CFG["run_tests"]:
    run_python(
        "Regression tests",
        "unittest",
        "discover",
        "-s",
        PROJECT_ROOT / "tests",
        "-v",
        module=True,
        cwd=PROJECT_ROOT,
    )
else:
    print("Regression tests skipped by configuration.")

## 6. Preprocess paired image-mask patches

Goi truc tiep `src/data/preprocess_95cloud.py`. Pipeline luu image-mask pair; nhan train duoc tinh lai tu dung crop mask trong `CloudDataset`.

In [ ]:
preprocessed_ready = (
    any((PROCESSED_ALL / "cloud").glob("*.npy"))
    or any((PROCESSED_ALL / "clear").glob("*.npy"))
) and any((PROCESSED_ALL / "masks").glob("*.npy"))

if CFG["run_preprocess"] and not (preprocessed_ready and not CFG["force_preprocess"]):
    arguments = [
        "--data_dir", DATA_ROOT,
        "--out_dir", PROCESSED_ALL,
        "--patch_size", CFG["source_patch_size"],
        "--cloud_ratio_threshold", CFG["cloud_ratio_threshold"],
        "--channels", CFG["channels"],
    ]
    if CFG["force_preprocess"]:
        arguments.append("--force")
    run_python("Preprocess 95-Cloud image-mask pairs", "src/data/preprocess_95cloud.py", *arguments)
elif preprocessed_ready:
    print("Using existing paired patches:", PROCESSED_ALL)
else:
    raise FileNotFoundError("Preprocessing is disabled but no processed image-mask pairs exist.")

## 7. Scene-level split

Goi `src/data/split_dataset.py`; tat ca patch va mask cua cung scene duoc giu trong mot split.

In [ ]:
SPLIT_MANIFEST = PROCESSED_ROOT / "scene_split_manifest.json"
split_ready = SPLIT_MANIFEST.is_file() and all(
    (PROCESSED_ROOT / split / "masks").is_dir() for split in ("train", "val", "test")
)

if CFG["run_split"] and not (split_ready and not CFG["force_split"]):
    arguments = [
        "--src_dir", PROCESSED_ALL,
        "--out_dir", PROCESSED_ROOT,
        "--val_ratio", CFG["val_ratio"],
        "--test_ratio", CFG["test_ratio"],
        "--seed", CFG["seed"],
    ]
    if CFG["force_split"]:
        arguments.append("--force")
    run_python("Scene-level split", "src/data/split_dataset.py", *arguments)
elif split_ready:
    print("Using existing scene split:", SPLIT_MANIFEST)
else:
    raise FileNotFoundError("Scene split is disabled but its manifest is missing.")

## 8. Validate dataset with `src/data/cloud_dataset.py`

Khong dinh nghia lai Dataset trong notebook. Cell nay import truc tiep `CloudDataset` tu codebase, doi chieu scene manifest va duyet toan bo ba split bang `DataLoader` truoc khi train.

In [ ]:
DATASET_SUMMARY_PATH = RESULTS_DIR / "dataset_validation.json"

if CFG["validate_dataset"]:
    src_path = str(PROJECT_ROOT / "src")
    if src_path not in sys.path:
        sys.path.insert(0, src_path)

    from data.cloud_dataset import CloudDataset
    from torch.utils.data import DataLoader
    from train import set_seed

    set_seed(CFG["seed"])

    split_manifest = json.loads(SPLIT_MANIFEST.read_text(encoding="utf-8"))
    required_splits = ("train", "val", "test")
    missing_splits = [split for split in required_splits if split not in split_manifest]
    if missing_splits:
        raise ValueError(f"Split manifest is missing entries: {missing_splits}")

    scene_sets = {split: set(split_manifest[split]["scenes"]) for split in required_splits}
    for left_index, left_split in enumerate(required_splits):
        for right_split in required_splits[left_index + 1:]:
            overlap = sorted(scene_sets[left_split] & scene_sets[right_split])
            if overlap:
                raise ValueError(
                    f"Scene leakage between {left_split} and {right_split}: {overlap[:5]}"
                )

    dataset_summary = {}
    for split in required_splits:
        manifest_entry = split_manifest[split]
        if not manifest_entry.get("pairing_valid", False):
            raise ValueError(f"Manifest reports invalid image-mask pairing for {split}")

        dataset = CloudDataset(
            PROCESSED_ROOT / split,
            is_train=split == "train",
            target_channels=CFG["channels"],
            channel_dropout_p=CFG["channel_dropout_p"],
            crop_size=CFG["crop_size"],
            cloud_ratio_threshold=CFG["cloud_ratio_threshold"],
        )
        expected_samples = int(manifest_entry["image_count"])
        if len(dataset) != expected_samples:
            raise ValueError(
                f"{split} sample count mismatch: dataset={len(dataset)}, manifest={expected_samples}"
            )

        loader = DataLoader(
            dataset,
            batch_size=CFG["dataset_validation_batch_size"],
            shuffle=False,
            num_workers=0,
        )
        checked_samples = 0
        positive_labels = 0
        value_min = float("inf")
        value_max = float("-inf")
        for images, labels in loader:
            expected_shape = (CFG["channels"], CFG["crop_size"], CFG["crop_size"])
            if images.ndim != 4 or tuple(images.shape[1:]) != expected_shape:
                raise ValueError(
                    f"Unexpected {split} batch shape {tuple(images.shape)}; "
                    f"expected N x {expected_shape[0]} x {expected_shape[1]} x {expected_shape[2]}"
                )
            if images.dtype != torch.float32:
                raise TypeError(f"Unexpected {split} image dtype: {images.dtype}")
            if not torch.isfinite(images).all():
                raise ValueError(f"Non-finite image value found in {split}")
            if labels.dtype != torch.float32 or not torch.all((labels == 0) | (labels == 1)):
                raise ValueError(f"Invalid labels found in {split}")

            checked_samples += int(images.shape[0])
            positive_labels += int(labels.sum().item())
            value_min = min(value_min, float(images.min().item()))
            value_max = max(value_max, float(images.max().item()))

        if checked_samples != expected_samples:
            raise ValueError(
                f"{split} DataLoader count mismatch: checked={checked_samples}, expected={expected_samples}"
            )
        dataset_summary[split] = {
            "scene_count": int(manifest_entry["scene_count"]),
            "samples": checked_samples,
            "positive_labels_in_validation_pass": positive_labels,
            "tensor_shape": [CFG["channels"], CFG["crop_size"], CFG["crop_size"]],
            "tensor_dtype": "float32",
            "value_range": [value_min, value_max],
        }

    with DATASET_SUMMARY_PATH.open("w", encoding="utf-8") as file_handle:
        json.dump(dataset_summary, file_handle, indent=2)
    print(json.dumps(dataset_summary, indent=2))
    print("Dataset validation saved:", DATASET_SUMMARY_PATH)
else:
    print("Dataset validation skipped by configuration.")

## 9. Train with `src/train.py`

Dynamic crop label, `pos_weight`, seed, AMP, model, training loop va W&B logging deu den tu pipeline chuan. API key duoc doc tu Kaggle Secret `WANDB_API_KEY`.

In [ ]:
if CFG["run_train"]:
    arguments = [
        "--train_dir", PROCESSED_ROOT / "train",
        "--val_dir", PROCESSED_ROOT / "val",
        "--out_dir", CHECKPOINT_DIR,
        "--epochs", CFG["epochs"],
        "--batch_size", CFG["batch_size"],
        "--lr", CFG["learning_rate"],
        "--channels", CFG["channels"],
        "--crop_size", CFG["crop_size"],
        "--cloud_ratio_threshold", CFG["cloud_ratio_threshold"],
        "--channel_dropout_p", CFG["channel_dropout_p"],
        "--num_workers", CFG["num_workers"],
        "--seed", CFG["seed"],
        "--pos_weight_samples_per_patch", CFG["pos_weight_samples_per_patch"],
        "--threshold", CFG["probability_threshold"],
    ]
    if not CFG["use_pos_weight"]:
        arguments.append("--no_pos_weight")
    if not CFG["amp"]:
        arguments.append("--no_amp")
    if CFG["use_wandb"]:
        arguments.extend([
            "--wandb",
            "--wandb_project", CFG["wandb_project"],
            "--wandb_mode", CFG["wandb_mode"],
        ])
        for option, value in (
            ("--wandb_entity", CFG["wandb_entity"]),
            ("--wandb_run_name", CFG["wandb_run_name"]),
            ("--wandb_group", CFG["wandb_group"]),
        ):
            if value:
                arguments.extend([option, value])
        if CFG["wandb_tags"]:
            arguments.extend(["--wandb_tags", *CFG["wandb_tags"]])
    run_python("Train 95-Cloud classifier", "src/train.py", *arguments)
else:
    print("Training skipped. Only a checkpoint created by src/train.py is accepted.")

if (CFG["run_eval"] or CFG["run_export"]) and not BEST_CHECKPOINT.is_file():
    raise FileNotFoundError(
        f"Required checkpoint not found: {BEST_CHECKPOINT}. Legacy notebook checkpoints are not accepted."
    )

## 10. Evaluate with `src/eval.py`

In [ ]:
if CFG["run_eval"]:
    eval_arguments = [
        "--test_dir", PROCESSED_ROOT / "test",
        "--model_path", BEST_CHECKPOINT,
        "--batch_size", CFG["batch_size"],
        "--channels", CFG["channels"],
        "--crop_size", CFG["crop_size"],
        "--cloud_ratio_threshold", CFG["cloud_ratio_threshold"],
        "--num_workers", CFG["num_workers"],
        "--seed", CFG["seed"],
        "--threshold", CFG["probability_threshold"],
    ]
    if CFG["use_wandb"]:
        eval_arguments.extend([
            "--wandb", "--wandb_project", CFG["wandb_project"],
            "--wandb_mode", CFG["wandb_mode"],
        ])
        if CFG["wandb_entity"]:
            eval_arguments.extend(["--wandb_entity", CFG["wandb_entity"]])
        wandb_metadata_path = CHECKPOINT_DIR / "wandb_run.json"
        if wandb_metadata_path.is_file():
            wandb_metadata = json.loads(wandb_metadata_path.read_text(encoding="utf-8"))
            if wandb_metadata.get("id"):
                eval_arguments.extend(["--wandb_run_id", wandb_metadata["id"]])
    run_python("Evaluate 95-Cloud classifier", "src/eval.py", *eval_arguments)
else:
    print("Evaluation skipped by configuration.")

## 11. Export with `src/export_onnx.py`

In [ ]:
if CFG["run_export"]:
    arguments = [
        "--model_path", BEST_CHECKPOINT,
        "--out_onnx", ONNX_PATH,
        "--batch_size", 1,
        "--channels", CFG["channels"],
        "--patch_size", CFG["crop_size"],
    ]
    if CFG["dynamic_batch_onnx"]:
        arguments.append("--dynamic_batch")
    run_python("Export ONNX", "src/export_onnx.py", *arguments)
else:
    print("ONNX export skipped by configuration.")

## 12. Provenance and artifacts

Luu Git commit, split manifest hash, seed, package/runtime version va artifact checksum. Day la provenance cua baseline, chua thay the checkpoint bundle/InputSpec trong P1.

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as file_handle:
        for chunk in iter(lambda: file_handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def git_value(*arguments):
    result = subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), *arguments],
        capture_output=True,
        text=True,
        check=False,
    )
    return result.stdout.strip() if result.returncode == 0 else "unavailable"

artifact_paths = [
    SPLIT_MANIFEST,
    CHECKPOINT_DIR / "training_config.json",
    CHECKPOINT_DIR / "wandb_run.json",
    BEST_CHECKPOINT,
    CHECKPOINT_DIR / "last_model.pth",
    DATASET_SUMMARY_PATH,
    RESULTS_DIR / "eval_metrics.json",
    ONNX_PATH,
]
artifact_records = {
    str(path.relative_to(RUN_ROOT)): {
        "bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    }
    for path in artifact_paths
    if path.is_file()
}

provenance = {
    "pipeline_entrypoint": "src/train.py",
    "checkpoint_origin": "src/train.py",
    "legacy_notebook_checkpoint_allowed": False,
    "radiometric_contract_status": "unverified_baseline_only",
    "cross_satellite_status": "blocked",
    "git_commit": git_value("rev-parse", "HEAD"),
    "git_status": git_value("status", "--short"),
    "split_manifest_sha256": sha256_file(SPLIT_MANIFEST) if SPLIT_MANIFEST.is_file() else None,
    "config": CFG,
    "runtime": {
        "python": platform.python_version(),
        "platform": platform.platform(),
        "torch": torch.__version__,
        "torchvision": importlib.metadata.version("torchvision"),
        "numpy": importlib.metadata.version("numpy"),
        "tifffile": importlib.metadata.version("tifffile"),
        "wandb": importlib.metadata.version("wandb"),
        "cuda_runtime": torch.version.cuda,
        "cudnn": torch.backends.cudnn.version(),
        "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "cudnn_benchmark": torch.backends.cudnn.benchmark,
        "cudnn_deterministic": torch.backends.cudnn.deterministic,
    },
    "artifacts": artifact_records,
}
PROVENANCE_PATH = RESULTS_DIR / "notebook_run_provenance.json"
with PROVENANCE_PATH.open("w", encoding="utf-8") as file_handle:
    json.dump(provenance, file_handle, indent=2)

print("Provenance:", PROVENANCE_PATH)
for relative_path, details in artifact_records.items():
    print(f"- {relative_path}: {details['bytes']:,} bytes, sha256={details['sha256'][:12]}...")

metrics_path = RESULTS_DIR / "eval_metrics.json"
if metrics_path.is_file():
    print("\nEvaluation summary:")
    print(json.dumps(json.loads(metrics_path.read_text(encoding="utf-8")), indent=2))